# Olist E-commerce Delivery Performance Analysis

**Goal:** Analyze delivery speed, late-delivery rates, and monthly trends using the Olist orders dataset.

**Workflow:** Load data → inspect data → prepare dates → calculate delivery KPIs → analyze monthly performance → visualize trends → export a clean CSV for Looker Studio.

**Dataset required:** `olist_orders_dataset.csv`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)

## 1. Load the data
Run this cell first. If the CSV is not in the Colab session, you can upload it when prompted.

In [ ]:
dataset_path = Path("olist_orders_dataset.csv")

# Check the usual Colab location too.
if not dataset_path.exists() and Path("/content/olist_orders_dataset.csv").exists():
    dataset_path = Path("/content/olist_orders_dataset.csv")

if not dataset_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("Please upload olist_orders_dataset.csv.")
        dataset_path = Path(csv_files[0])
    except ImportError as exc:
        raise FileNotFoundError(
            "Place olist_orders_dataset.csv in the same folder as this notebook, then rerun."
        ) from exc

df = pd.read_csv(dataset_path)
print(f"Loaded file: {dataset_path}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

## 2. Take a first look at the data

In [ ]:
display(df.head())
print("\nColumn information:")
df.info()

print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

if "order_status" in df.columns:
    print("\nOrder status distribution:")
    display(df["order_status"].value_counts(dropna=False))

## 3. Prepare the delivery data
For the delivery calculations, I use orders with the required dates available. An order counts as late if it arrived after its estimated delivery date.

In [ ]:
required_columns = [
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

orders = df.copy()
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

delivered = orders.loc[
    orders["order_status"].eq("delivered")
    & orders["order_purchase_timestamp"].notna()
    & orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
].copy()

delivered["delivery_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]
).dt.total_seconds() / 86400

delivered["delay_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

delivered["is_late"] = delivered["delay_days"] > 0
delivered["purchase_month"] = delivered["order_purchase_timestamp"].dt.to_period("M")

print(f"Delivered orders with all required dates: {len(delivered):,}")
display(delivered[[
    "order_id", "order_purchase_timestamp", "order_delivered_customer_date",
    "order_estimated_delivery_date", "delivery_days", "delay_days", "is_late"
]].head())

## 4. Overall delivery numbers

In [ ]:
total_orders = len(delivered)
late_orders_count = int(delivered["is_late"].sum())
on_time_or_early = total_orders - late_orders_count

kpis = pd.DataFrame({
    "Metric": [
        "Delivered orders analyzed",
        "Late deliveries",
        "On-time or early deliveries",
        "Late delivery rate (%)",
        "On-time or early rate (%)",
        "Average delivery time (days)",
        "Average delay among late orders (days)",
    ],
    "Value": [
        total_orders,
        late_orders_count,
        on_time_or_early,
        round(late_orders_count / total_orders * 100, 2) if total_orders else 0,
        round(on_time_or_early / total_orders * 100, 2) if total_orders else 0,
        round(delivered["delivery_days"].mean(), 2) if total_orders else None,
        round(delivered.loc[delivered["is_late"], "delay_days"].mean(), 2)
            if late_orders_count else 0,
    ],
})
display(kpis)

## 5. Delivery performance by month
The monthly groups are based on the order purchase month.

In [ ]:
monthly = delivered.groupby("purchase_month").agg(
    delivered_orders=("order_id", "nunique"),
    late_orders=("is_late", "sum"),
    avg_delivery_days=("delivery_days", "mean"),
).reset_index()

monthly["month"] = monthly["purchase_month"].dt.to_timestamp()
monthly["late_rate_percent"] = (
    monthly["late_orders"] / monthly["delivered_orders"] * 100
).round(2)
monthly["avg_delivery_days"] = monthly["avg_delivery_days"].round(2)

monthly = monthly[[
    "month", "delivered_orders", "late_orders",
    "late_rate_percent", "avg_delivery_days"
]].sort_values("month").reset_index(drop=True)

display(monthly.tail(15))

## 6. Plot the late-delivery rate

In [ ]:
if not monthly.empty:
    plt.figure(figsize=(12, 5))
    plt.plot(monthly["month"], monthly["late_rate_percent"], marker="o")
    plt.title("Monthly Late-Delivery Rate")
    plt.xlabel("Purchase Month")
    plt.ylabel("Late Deliveries (%)")
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No monthly data is available to plot.")

## 7. Check which months had higher late-delivery rates
I only compare months with at least 100 delivered orders so that months with very few orders do not dominate the result.

In [ ]:
monthly_check = monthly.loc[monthly["delivered_orders"] >= 100].copy()
highest_late_months = monthly_check.sort_values(
    "late_rate_percent", ascending=False
).head(5)

print("Five months with the highest late-delivery rates (minimum 100 orders):")
display(highest_late_months)

if not monthly_check.empty:
    plt.figure(figsize=(12, 5))
    plt.plot(monthly_check["month"], monthly_check["late_rate_percent"], marker="o")
    plt.title("Monthly Late-Delivery Rate (Months with at Least 100 Orders)")
    plt.xlabel("Purchase Month")
    plt.ylabel("Late Deliveries (%)")
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No months meet the 100-order threshold.")

## 8. Save the monthly summary
This CSV can be connected to Looker Studio. The month is saved in a date format so it is easier to use in charts.

In [ ]:
output_path = Path("olist_monthly_summary.csv")
export_monthly = monthly.copy()
export_monthly["month"] = export_monthly["month"].dt.strftime("%Y-%m-%d")
export_monthly.to_csv(output_path, index=False)

print(f"Saved: {output_path.resolve()}")
print(f"Monthly records exported: {len(export_monthly)}")
display(export_monthly.head())

## 9. Download the CSV (optional)
Run this cell in Google Colab if you want to download the summary.

In [ ]:
try:
    from google.colab import files
    files.download("olist_monthly_summary.csv")
except ImportError:
    print("Not running in Google Colab. The CSV is saved in the notebook's working folder.")